# HF Transformers VLM Text-Generation Attack

This notebook converts `attack_qwen25vl_text_generation.py` into an interactive workflow for a targeted PGD attack against supported Hugging Face vision-language models. It perturbs a source image so the model is pushed toward generating a fixed target caption for a fixed user prompt.

Supported model families:
- Qwen VL checkpoints with `model_type` `qwen2_vl` or `qwen2_5_vl`
- Gemma 3 checkpoints with `model_type` `gemma3`
- Fixed-resolution LLaVA checkpoints with `model_type` `llava`

Runtime assumptions:
- Launch the notebook from this repository or one of its subdirectories.
- A CUDA device is available.
- The selected supported model weights are accessible to `transformers`.

Expected outputs:
- `results/<model_slug>_textgen_adv.png`
- `results/<model_slug>_textgen_noise.png`
- `results/<model_slug>_textgen_generations.txt`


In [ ]:
from pathlib import Path

import torch
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
from transformers import AutoModelForImageTextToText, AutoProcessor


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the repo root from the current working directory. "
        "Launch the notebook from this repository or one of its subdirectories."
    )


REPO_ROOT = find_repo_root()
RESULTS_DIR = REPO_ROOT / "results"

print(f"Repo root: {REPO_ROOT}")
print(f"Results dir: {RESULTS_DIR}")


In [ ]:
MODEL_NAME = "google/gemma-3-4b-it" # "Qwen/Qwen2.5-VL-7B-Instruct" or "google/gemma-3-4b-it"
MODEL_FAMILY = "auto"  # "auto", "qwen", "llava", or "gemma"
SOURCE_IMAGE_PATH = REPO_ROOT / "data" / "images" / "worksheet_000002.png"
USER_PROMPT = "Solve the math problems shown in the image. Show steps and give the final answers."
TARGET_TEXTS: list[str] = [
    "The image shows a toaster",
    "A toaster is sitting on a clean kitchen counter.",
    "Toaster.",
]

EPSILON = 64 / 255
ALPHA = 1 / 255
STEPS = 4000
ATTACK_IMAGE_SIZE = (400, 400)
MODEL_INPUT_SIZE = 448  # Used only for the Qwen family branch.
MAX_NEW_TOKENS = 128

MODEL_SLUG = "".join(char.lower() if char.isalnum() else "_" for char in MODEL_NAME).strip("_")
RESULT_PREFIX = f"{MODEL_SLUG}_textgen"
OUTPUT_ADV_PATH = RESULTS_DIR / f"{RESULT_PREFIX}_adv.png"
OUTPUT_NOISE_PATH = RESULTS_DIR / f"{RESULT_PREFIX}_noise.png"
OUTPUT_REPORT_PATH = RESULTS_DIR / f"{RESULT_PREFIX}_generations.txt"

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

INSPECTION_PROMPT = "What is shown in this image? Describe it in detail."

print(f"Model: {MODEL_NAME}")
print(f"Model family setting: {MODEL_FAMILY}")
print(f"Source image: {SOURCE_IMAGE_PATH}")
print(f"Prompt: {USER_PROMPT}")
print("Target texts:")
for target_text in TARGET_TEXTS:
    print(f"- {target_text}")


In [ ]:
def load_image_tensor(image_path: str | Path, device: torch.device, image_size: tuple[int, int]) -> torch.Tensor:
    image = Image.open(image_path).convert("RGB")
    image = image.resize(image_size, Image.Resampling.BICUBIC)
    return transforms.ToTensor()(image).to(device).unsqueeze(0)


SUPPORTED_MODEL_TYPES = {
    "gemma3": "gemma",
    "qwen2_vl": "qwen",
    "qwen2_5_vl": "qwen",
    "llava": "llava",
}

TOKEN_TYPE_INPUT_KEYS = ("mm_token_type_ids", "token_type_ids")


def resolve_model_family(requested_model_family: str, model_type: str) -> str:
    if requested_model_family not in {"auto", "gemma", "qwen", "llava"}:
        raise ValueError("MODEL_FAMILY must be one of: 'auto', 'gemma', 'qwen', 'llava'.")

    detected_model_family = SUPPORTED_MODEL_TYPES.get(model_type)
    if detected_model_family is None:
        supported_model_types = ", ".join(sorted(SUPPORTED_MODEL_TYPES))
        raise ValueError(
            f"Unsupported model type {model_type!r}. "
            f"This notebook supports model types: {supported_model_types}."
        )

    if requested_model_family == "auto":
        return detected_model_family

    if requested_model_family != detected_model_family:
        raise ValueError(
            f"MODEL_FAMILY={requested_model_family!r} does not match model type {model_type!r}. "
            f"Use MODEL_FAMILY={detected_model_family!r} or 'auto'."
        )

    return requested_model_family


def pack_for_qwen(
    image_tensor: torch.Tensor,
    *,
    model_input_size: int,
    mean: torch.Tensor,
    std: torch.Tensor,
    patch_size: int,
    temporal_patch_size: int,
    merge_size: int,
    device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor]:
    height, width = image_tensor.shape[-2:]
    scale = min(model_input_size / height, model_input_size / width)
    resized_h = max(1, int(round(height * scale)))
    resized_w = max(1, int(round(width * scale)))

    x = F.interpolate(
        image_tensor.unsqueeze(0),
        size=(resized_h, resized_w),
        mode="bilinear",
        align_corners=False,
    ).squeeze(0)

    pad_h = model_input_size - resized_h
    pad_w = model_input_size - resized_w
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left
    x = F.pad(x, (pad_left, pad_right, pad_top, pad_bottom), value=1.0)
    x = (x - mean) / std

    frames = x.unsqueeze(0)
    if frames.shape[0] % temporal_patch_size != 0:
        repeats = temporal_patch_size - (frames.shape[0] % temporal_patch_size)
        frames = torch.cat([frames, frames[-1:].repeat(repeats, 1, 1, 1)], dim=0)

    channels = frames.shape[1]
    grid_t = frames.shape[0] // temporal_patch_size
    grid_h = frames.shape[2] // patch_size
    grid_w = frames.shape[3] // patch_size

    patches = frames.reshape(
        grid_t,
        temporal_patch_size,
        channels,
        grid_h // merge_size,
        merge_size,
        patch_size,
        grid_w // merge_size,
        merge_size,
        patch_size,
    )
    patches = patches.permute(0, 3, 6, 4, 7, 2, 1, 5, 8)

    pixel_values = patches.reshape(
        grid_t * grid_h * grid_w,
        channels * temporal_patch_size * patch_size * patch_size,
    )
    image_grid_thw = torch.tensor([[grid_t, grid_h, grid_w]], device=device, dtype=torch.long)
    return pixel_values, image_grid_thw


def pack_for_llava(
    image_tensor: torch.Tensor,
    *,
    shortest_edge: int,
    crop_size: tuple[int, int],
    mean: torch.Tensor,
    std: torch.Tensor,
) -> torch.Tensor:
    height, width = image_tensor.shape[-2:]
    scale = shortest_edge / min(height, width)
    resized_h = max(crop_size[0], int(round(height * scale)))
    resized_w = max(crop_size[1], int(round(width * scale)))

    x = F.interpolate(
        image_tensor.unsqueeze(0),
        size=(resized_h, resized_w),
        mode="bilinear",
        align_corners=False,
    )

    crop_h, crop_w = crop_size
    top = max(0, (resized_h - crop_h) // 2)
    left = max(0, (resized_w - crop_w) // 2)
    x = x[:, :, top : top + crop_h, left : left + crop_w]
    return (x - mean) / std


def pack_for_gemma(
    image_tensor: torch.Tensor,
    *,
    size: tuple[int, int],
    rescale_factor: float,
    mean: torch.Tensor,
    std: torch.Tensor,
) -> torch.Tensor:
    x = F.interpolate(
        image_tensor.unsqueeze(0),
        size=size,
        mode="bilinear",
        align_corners=False,
    )

    if torch.max(x) > 1.0:
        x = x * rescale_factor

    return (x - mean) / std


def save_noise_visualization(delta: torch.Tensor, output_path: Path) -> None:
    noise = torch.clamp(delta.squeeze(0).cpu() * 10 + 0.5, 0.0, 1.0)
    transforms.ToPILImage()(noise).save(output_path)


def show_saved_results(image_paths: tuple[Path, ...], report_path: Path) -> None:
    for path in image_paths:
        print(path)
        if path.exists():
            display(Image.open(path))
        else:
            print("Missing output file.")

    print(report_path)
    if report_path.exists():
        print(report_path.read_text())
    else:
        print("Missing report file.")


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA device is required for this notebook.")

device = torch.device("cuda")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"[Info] Loading model: {MODEL_NAME}")
processor = AutoProcessor.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
).to(device)
model.config.use_cache = False
model.eval()
model.requires_grad_(False)

MODEL_FAMILY_RESOLVED = resolve_model_family(MODEL_FAMILY, model.config.model_type)
print(
    f"[Info] Resolved model family: {MODEL_FAMILY_RESOLVED} "
    f"(model_type={model.config.model_type})"
)

x_clean = load_image_tensor(SOURCE_IMAGE_PATH, device, ATTACK_IMAGE_SIZE)

prompt_processor_kwargs = {}

if MODEL_FAMILY_RESOLVED == "qwen":
    vision_config = model.config.vision_config
    patch_size = vision_config.patch_size
    temporal_patch_size = vision_config.temporal_patch_size
    merge_size = vision_config.spatial_merge_size
    mean = torch.tensor(CLIP_MEAN, device=device)
    std = torch.tensor(CLIP_STD, device=device)
    dummy_image_size = (MODEL_INPUT_SIZE, MODEL_INPUT_SIZE)
elif MODEL_FAMILY_RESOLVED == "gemma":
    image_processor = processor.image_processor
    gemma_size = (
        int(image_processor.size["height"]),
        int(image_processor.size["width"]),
    )
    gemma_rescale_factor = float(image_processor.rescale_factor)
    mean = torch.tensor(image_processor.image_mean, device=device)
    std = torch.tensor(image_processor.image_std, device=device)
    dummy_image_size = (gemma_size[1], gemma_size[0])
    prompt_processor_kwargs["do_pan_and_scan"] = False
else:
    image_processor = processor.image_processor
    llava_shortest_edge = image_processor.size.get("shortest_edge")
    if llava_shortest_edge is None:
        llava_shortest_edge = min(image_processor.size["height"], image_processor.size["width"])
    llava_shortest_edge = int(llava_shortest_edge)
    llava_crop_size = (
        int(image_processor.crop_size["height"]),
        int(image_processor.crop_size["width"]),
    )
    mean = torch.tensor(image_processor.image_mean, device=device)
    std = torch.tensor(image_processor.image_std, device=device)
    dummy_image_size = (llava_crop_size[1], llava_crop_size[0])


def build_vision_inputs(image_tensor: torch.Tensor) -> dict[str, torch.Tensor]:
    if MODEL_FAMILY_RESOLVED == "qwen":
        pixel_values, image_grid_thw = pack_for_qwen(
            image_tensor,
            model_input_size=MODEL_INPUT_SIZE,
            mean=mean.view(3, 1, 1),
            std=std.view(3, 1, 1),
            patch_size=patch_size,
            temporal_patch_size=temporal_patch_size,
            merge_size=merge_size,
            device=device,
        )
        return {
            "pixel_values": pixel_values,
            "image_grid_thw": image_grid_thw,
        }

    if MODEL_FAMILY_RESOLVED == "gemma":
        pixel_values = pack_for_gemma(
            image_tensor,
            size=gemma_size,
            rescale_factor=gemma_rescale_factor,
            mean=mean.view(1, 3, 1, 1),
            std=std.view(1, 3, 1, 1),
        )
        return {"pixel_values": pixel_values}

    pixel_values = pack_for_llava(
        image_tensor,
        shortest_edge=llava_shortest_edge,
        crop_size=llava_crop_size,
        mean=mean.view(1, 3, 1, 1),
        std=std.view(1, 3, 1, 1),
    )
    return {"pixel_values": pixel_values}


print(f"Loaded source image tensor with shape: {tuple(x_clean.shape)}")


In [ ]:
def build_text_model_inputs(prompt_inputs: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    model_inputs = {
        "input_ids": prompt_inputs["input_ids"].to(device),
        "attention_mask": prompt_inputs["attention_mask"].to(device),
    }
    for token_type_key in TOKEN_TYPE_INPUT_KEYS:
        token_type_ids = prompt_inputs.get(token_type_key)
        if token_type_ids is not None:
            model_inputs[token_type_key] = token_type_ids.to(device)
    return model_inputs


def build_prompt_inputs(prompt: str) -> tuple[str, dict[str, torch.Tensor]]:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    rendered_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    dummy_image = Image.new("RGB", dummy_image_size, color="white")
    prompt_inputs = processor(
        text=[rendered_prompt],
        images=[dummy_image],
        return_tensors="pt",
        **prompt_processor_kwargs,
    )
    return rendered_prompt, build_text_model_inputs(prompt_inputs)


prompt_text, prompt_model_inputs = build_prompt_inputs(USER_PROMPT)
prompt_input_ids = prompt_model_inputs["input_ids"]

tokenizer = processor.tokenizer
if not TARGET_TEXTS or any(not isinstance(target_text, str) or not target_text for target_text in TARGET_TEXTS):
    raise ValueError("TARGET_TEXTS must be a non-empty list of non-empty strings.")

eos_token_id = tokenizer.eos_token_id
target_batches = []
for target_text in TARGET_TEXTS:
    target_ids = tokenizer(target_text, add_special_tokens=False, return_tensors="pt")["input_ids"].to(device)
    if eos_token_id is not None and (target_ids.shape[1] == 0 or target_ids[0, -1].item() != eos_token_id):
        target_ids = torch.cat([target_ids, torch.tensor([[eos_token_id]], device=device)], dim=1)

    full_model_inputs = {
        "input_ids": torch.cat([prompt_model_inputs["input_ids"], target_ids], dim=1),
        "attention_mask": torch.cat([prompt_model_inputs["attention_mask"], torch.ones_like(target_ids)], dim=1),
    }
    for token_type_key in TOKEN_TYPE_INPUT_KEYS:
        token_type_ids = prompt_model_inputs.get(token_type_key)
        if token_type_ids is not None:
            full_model_inputs[token_type_key] = torch.cat(
                [
                    token_type_ids,
                    torch.zeros(
                        target_ids.shape,
                        device=device,
                        dtype=token_type_ids.dtype,
                    ),
                ],
                dim=1,
            )

    labels = full_model_inputs["input_ids"].clone()
    labels[:, : prompt_model_inputs["input_ids"].shape[1]] = -100
    target_batches.append(
        {
            "target_text": target_text,
            "model_inputs": full_model_inputs,
            "labels": labels,
        }
    )


def target_score(target_batch: dict, vision_inputs: dict[str, torch.Tensor]) -> torch.Tensor:
    outputs = model(
        **target_batch["model_inputs"],
        **vision_inputs,
        use_cache=False,
        return_dict=True,
    )
    shifted_logits = outputs.logits[:, :-1, :]
    shifted_labels = target_batch["labels"][:, 1:]
    valid_mask = shifted_labels != -100
    safe_labels = shifted_labels.masked_fill(~valid_mask, 0)
    token_log_probs = F.log_softmax(shifted_logits, dim=-1)
    token_log_probs = token_log_probs.gather(dim=-1, index=safe_labels.unsqueeze(-1)).squeeze(-1)
    token_mask = valid_mask.to(token_log_probs.dtype)
    avg_nll = -(token_log_probs * token_mask).sum() / token_mask.sum()
    return -avg_nll


def target_loss(vision_inputs: dict[str, torch.Tensor], *, backward: bool = False) -> torch.Tensor:
    with torch.no_grad():
        detached_scores = torch.stack([target_score(target_batch, vision_inputs) for target_batch in target_batches])
        aggregate_loss = -(
            torch.logsumexp(detached_scores, dim=0) - detached_scores.new_tensor(len(target_batches)).log()
        )

    if not backward:
        return aggregate_loss

    weights = torch.softmax(detached_scores, dim=0)
    pixel_values_grad = torch.zeros_like(vision_inputs["pixel_values"])
    for weight, target_batch in zip(weights, target_batches):
        pixel_values_ref = vision_inputs["pixel_values"].detach().requires_grad_(True)
        vision_inputs_ref = dict(vision_inputs)
        vision_inputs_ref["pixel_values"] = pixel_values_ref
        score = target_score(target_batch, vision_inputs_ref)
        grad = torch.autograd.grad(-weight * score, pixel_values_ref)[0]
        pixel_values_grad.add_(grad)

    vision_inputs["pixel_values"].backward(pixel_values_grad)
    return aggregate_loss


def generate_from_image(image_tensor: torch.Tensor) -> str:
    vision_inputs = build_vision_inputs(image_tensor)
    generated = model.generate(
        **prompt_model_inputs,
        **vision_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
    )
    new_tokens = generated[:, prompt_input_ids.shape[1] :]
    return processor.batch_decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()


print(prompt_text)


In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with torch.no_grad():
    clean_vision_inputs = build_vision_inputs(x_clean.squeeze(0))
    clean_loss = target_loss(clean_vision_inputs).item()

delta = torch.zeros_like(x_clean, requires_grad=True)

print("[Info] Starting PGD text-generation attack...")
progress = tqdm(range(STEPS))
for _ in progress:
    delta.grad = None

    x_adv = torch.clamp(x_clean + delta, 0.0, 1.0)
    vision_inputs = build_vision_inputs(x_adv.squeeze(0))
    loss = target_loss(vision_inputs, backward=True)

    with torch.no_grad():
        delta -= ALPHA * delta.grad.sign()
        delta.clamp_(-EPSILON, EPSILON)
        delta.copy_(torch.clamp(x_clean + delta, 0.0, 1.0) - x_clean)

    progress.set_postfix(loss=f"{loss.item():.4f}")

x_final = torch.clamp(x_clean + delta, 0.0, 1.0)

with torch.no_grad():
    adv_vision_inputs = build_vision_inputs(x_final.squeeze(0))
    adv_loss = target_loss(adv_vision_inputs).item()
    clean_output = generate_from_image(x_clean.squeeze(0))
    adv_output = generate_from_image(x_final.squeeze(0))

transforms.ToPILImage()(x_final.squeeze(0).cpu()).save(OUTPUT_ADV_PATH)
save_noise_visualization(delta, OUTPUT_NOISE_PATH)
OUTPUT_REPORT_PATH.write_text(
    "\n".join(
        [
            f"Prompt: {prompt_text}",
            "Target texts:",
            *[f"- {target_text}" for target_text in TARGET_TEXTS],
            "",
            f"Clean aggregate target loss: {clean_loss:.6f}",
            f"Adversarial aggregate target loss: {adv_loss:.6f}",
            "",
            "Clean generation:",
            clean_output,
            "",
            "Adversarial generation:",
            adv_output,
        ]
    )
)

print(f"[Info] Clean aggregate target loss: {clean_loss:.6f}")
print(f"[Info] Adversarial aggregate target loss: {adv_loss:.6f}")
print("\n===== CLEAN GENERATION =====\n")
print(clean_output)
print("\n===== ADVERSARIAL GENERATION =====\n")
print(adv_output)
print(f"\n[Success] Saved adversarial image to {OUTPUT_ADV_PATH}")
print(f"[Success] Saved perturbation visualization to {OUTPUT_NOISE_PATH}")
print(f"[Success] Saved text report to {OUTPUT_REPORT_PATH}")


In [ ]:
show_saved_results((OUTPUT_ADV_PATH, ), OUTPUT_REPORT_PATH)

# Inference

Pass the adversarial image through the selected VLM and display the output.


In [ ]:
inspection_prompt_text, inspection_model_inputs = build_prompt_inputs(INSPECTION_PROMPT)
vision_inputs = build_vision_inputs(x_final.squeeze(0))
generated = model.generate(
    **inspection_model_inputs,
    **vision_inputs,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
)
new_tokens = generated[:, inspection_model_inputs["input_ids"].shape[1] :]
adv_output_text = processor.batch_decode(
    new_tokens,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0].strip()

print(adv_output_text)
